# 1. Context

This notebook analyzes OCR performance of Google Document AI Solution over synthetic generated PDF images across variois degradation levels

# 2. Imports

In [1]:
import pandas as pd
from pathlib import Path
from collections import defaultdict
import plotly.express as px

In [2]:
import sys

In [3]:
notebook_path = Path()
sys.path.append(str(notebook_path.resolve().parent))

In [4]:
from src.viz.helper import display_box_plot

# 3. Utils

In [ ]:
# get csv paths for all language
results_root = Path("../results/google")
results_langs = [x.name.lower() for x in results_root.glob("*") if x.is_dir()]

In [6]:
writing_system_to_language = {'Devanagari': ['hindi','sanskrit','nepali','konkani'],
                              'tamil': ['tamil'],'telugu': ['telugu'],'Kannada': ['kannada'],'Malayalam': ['malayalam'],
                              'Bengali': ['bengali', 'assamese', 'manipuri'],'Meetei-mayek': ['manipuri'],'Gujarati': ['gujarati'],
                              'Gurmukhi': ['punjabi'],'Odia': ['oriya'],'Arabic': ['kashmiri', 'sindhi', 'urdu'],
                              'Latin': ['english'],'Ol-chiki': ['santali']}

In [8]:
language_to_writing_system = {
    "marathi": ["Devanagari"], "hindi": ["Devanagari"], "sanskrit": ["Devanagari"],
    "tamil": ["tamil"], "telugu": ["telugu"], "kannada": ["Kannada"],"malayalam": ["Malayalam"],
    "bengali": ["Bengali"], "assamese": ["Bengali"],"manipuri": ["Bengali", "Meetei-mayek"],"nepali": ["Devanagari"],
    "gujarati": ["Gujarati"], "punjabi": ["Gurmukhi"], "konkani": ["Devanagari"],"oriya": ["Odia"],"kashmiri": ["Devanagari", "Arabic"], 
    "sindhi": ["Arabic", "Devanagari"], "urdu": ["Arabic"],"english": ["Latin"], "santali": ["Ol-chiki", "Devanagari"],
    }

In [9]:
writing_sys_dict = defaultdict(list)

for language_res in results_langs:
    script = language_to_writing_system.get(language_res)[0]
    writing_sys_dict[script].append(language_res)

In [10]:
script_language_result = pd.Series(writing_sys_dict).to_frame(name='languages')
script_language_result.index.name = 'script'

## 3.1. Results Available Across Script and Language Combination

| script     | languages                                  |
|:-----------|:-------------------------------------------|
| Gujarati   | ['gujarati']                               |
| Gurmukhi   | ['punjabi']                                |
| Devanagari | ['hindi', 'konkani', 'sanskrit', 'nepali'] |
| telugu     | ['telugu']                                 |
| Bengali    | ['assamese', 'bengali']                    |
| Latin      | ['english']                                |
| Kannada    | ['kannada']                                |
| tamil      | ['tamil']                                  |
| Arabic     | ['sindhi']                                 |

In [13]:
def get_language_results_path(script: str, script_language_result: pd.DataFrame) -> list[Path]:
    """Get list of results path for given script"""

    results_script_lang = script_language_result.loc[script].to_list()[0]
    results_script_lang_path = [results_root.joinpath(lang).joinpath('results.csv') for lang in results_script_lang]

    return results_script_lang_path

In [20]:
def get_script_results(script: str, script_lang_df: pd.DataFrame):
    """Get results for a given script. output contains results for languages in the script"""

    results_script_path = get_language_results_path(script, script_lang_df)

    results_list = []
    for result_path in results_script_path:
        lang = result_path.parent.name
        df_res = pd.read_csv(result_path)
        df_res['language'] = lang
        results_list.append(df_res)

    results_script = pd.concat(results_list)
    results_script['script'] = script
    return results_script
    

## 3.2. Getting All the results

In [35]:
script_results_avail = script_language_result.index
results_consolidated = []
for script in script_results_avail:
    results_script = get_script_results(script=script, script_lang_df=script_language_result)
    results_consolidated.append(results_script)
consolidated_df = pd.concat(results_consolidated)

In [110]:
agg_results = (consolidated_df.groupby(['script', 'language']).agg(CER_AVG_L0=('cer_l0', 'median'),
                                                    CER_AVG_L1=('cer_l1', 'median'),
                                                    CER_AVG_L2=('cer_l2', 'median'),
                                                    CER_AVG_L3=('cer_l3', 'median'),
                                                    WER_AVG_L0=('wer_l0', 'median'),
                                                    WER_AVG_L1=('wer_l1', 'median'),
                                                    WER_AVG_L2=('wer_l2', 'median'),
                                                    WER_AVG_L3=('wer_l3', 'median')
                                                    ).round(3))

In [116]:
agg_results.columns = agg_results.columns.str.upper()

In [111]:
agg_results.to_clipboard(index=True) 

In [114]:
col_ord = ['file_id', 'language', 'script','ground_truth', 'ocr_output_L_0', 'ocr_output_L_1',
       'ocr_output_L_2', 'ocr_output_L_3', 'cer_l0', 'cer_l1', 'cer_l2',
       'cer_l3', 'wer_l0', 'wer_l1', 'wer_l2', 'wer_l3' ]
consolidated_df = consolidated_df[col_ord]

## upper casing column names
consolidated_df.columns = consolidated_df.columns.str.upper()

In [115]:
consolidated_df.round(3).to_clipboard(index=False)

# 4. Results Across Various Writing system (samples)

# 4.1. Devanagari 

In [21]:
results_devanagari = get_script_results(script='Devanagari', script_lang_df=script_language_result)

## 4.1.1. Box Plot Visualisation

In [16]:
display_box_plot(results_df=results_devanagari, script=script, metric_type='CER').show()

In [24]:
display_box_plot(results_df=results_devanagari, script=script, metric_type='WER').show()

## 5.1. Bengali

In [28]:
script = 'Bengali'
results_bengali = get_script_results(script='Bengali', script_lang_df=script_language_result)

### 5.1.1. Box Plot Visualisation

In [29]:
display_box_plot(results_df=results_bengali, script=script, metric_type='CER').show()

In [30]:
display_box_plot(results_df=results_bengali, script=script, metric_type='WER').show()

# 6. Addendum

## 6.1. Assessing High WER in Kannada

In [44]:
import jiwer

In [39]:
script = 'Kannada'
results_kn = get_script_results(script=script, script_lang_df=script_language_result)

In [50]:
idx = 0
gt = results_kn.loc[idx]['ground_truth']
ocred = results_kn.loc[idx]['ocr_output_L_0']

In [51]:
output_wer = jiwer.process_words(gt, ocred)

In [88]:
print(jiwer.visualize_alignment(output_wer, line_width=20))

=== SENTENCE 1 ===

REF: ನಿಮಿಷಕ್ಕೆ
HYP: ನಿಮಿಷಕ್ಕೆ
              

REF: ಕೇಂದ್ರ.
HYP: ಕೇಂದ್ರ,
           S

REF: ಸ್ಫೋಟಕ"ವೆಂದು
HYP: ಸ್ಫೋಟಕ"ವೆಂದು
                 

REF: ಸಾಂಕೇತಿಕ
HYP: ಸಾಂಕೇತಿಕ
             

REF: ಅಣುಸ್ಥಾವರವಾಗಿ
HYP: ಅಣುಸ್ಥಾವರವಾಗಿ
                  

REF: ಪೋಖ್ರಾನ್
HYP: ಪೋಗ್ರಾನ್
            S

REF: (ಪೋಕರಾನ್ ಎಂದೂ
HYP: (ಪೋಕರಾನ್ ಎಂದೂ
                  

REF: ಕರೆಯಲಾಗುತ್ತದೆ)
HYP: ಕರೆಯಲಾಗುತ್ತದೆ)
                   

REF: ಭಾರತದ ರಾಜ್ಯವಾದ
HYP: ಭಾರತದ ರಾಜ್ಯವಾದ
                   

REF: ರಾಜಾಸ್ಥಾನದಲ್ಲಿರುವ
HYP: ರಾಜಾಸ್ಥಾನದಲ್ಲಿರುವ
                      

REF:  ಜೈಸಾಲ್ಮರ್
HYP: ಜೈಸಾಲ್ಕ‌ರ್
              S

REF: ಜಿಲ್ಲೆಯಲ್ಲಿರುವ
HYP: ಜಿಲ್ಲೆಯಲ್ಲಿರುವ
                   

REF: ಒಂದು
HYP: ಒಂದು
         

REF: ಮುನಿಸಿಪಾಲಿಟಿ
HYP: ಮುನಿಸಿಪಾಲಿಟಿ
                 

REF: ಹಾಗೂ ನಗರವಾಗಿದೆ.
HYP: ಹಾಗೂ ನಗರವಾಗಿದೆ.
                    

REF: ಇದು ಥಾರ್
HYP: ಇದು ಥಾರ್
             

REF: ಮರುಭೂಮಿಯ ಒಂದು
HYP: ಮರುಭೂಮಿಯ ಒಂದು
                  

REF: ಮೂಲೆಯಲ್ಲಿದೆ
HYP:  ಮೊಲೆಯಲ್ಲಿದ
               S

REF: ಹಾಗೂ ಭಾರತದ ಮೊದಲ
HYP: ಹಾಗೂ ಭ

In [54]:
output_wer.alignments

[[AlignmentChunk(type='equal', ref_start_idx=0, ref_end_idx=1, hyp_start_idx=0, hyp_end_idx=1),
  AlignmentChunk(type='substitute', ref_start_idx=1, ref_end_idx=2, hyp_start_idx=1, hyp_end_idx=2),
  AlignmentChunk(type='equal', ref_start_idx=2, ref_end_idx=5, hyp_start_idx=2, hyp_end_idx=5),
  AlignmentChunk(type='substitute', ref_start_idx=5, ref_end_idx=6, hyp_start_idx=5, hyp_end_idx=6),
  AlignmentChunk(type='equal', ref_start_idx=6, ref_end_idx=12, hyp_start_idx=6, hyp_end_idx=12),
  AlignmentChunk(type='substitute', ref_start_idx=12, ref_end_idx=13, hyp_start_idx=12, hyp_end_idx=13),
  AlignmentChunk(type='equal', ref_start_idx=13, ref_end_idx=22, hyp_start_idx=13, hyp_end_idx=22),
  AlignmentChunk(type='substitute', ref_start_idx=22, ref_end_idx=23, hyp_start_idx=22, hyp_end_idx=23),
  AlignmentChunk(type='equal', ref_start_idx=23, ref_end_idx=50, hyp_start_idx=23, hyp_end_idx=50),
  AlignmentChunk(type='insert', ref_start_idx=50, ref_end_idx=50, hyp_start_idx=50, hyp_end_idx=51